# Week 4 Exercise — Game Engine Docstring Writer

**Week 4 focus:** Use a frontier model for code-related tasks (docstrings, unit tests, etc.).

**This solution:** A tool that helps **game programmers** (Unity, Pygame, Cocos2d-x) **write docstrings** for their code in each engine’s style. You paste a **function, method, or class** — not only a single function — and get back the same code with docstrings and brief comments.

1. **Docstrings & comments** — Paste code (one function, several methods, or a whole class like a MonoBehaviour). Get docstrings in the right format for your framework (PEP-257 for Pygame, XML docs for Unity C#, Doxygen-style for Cocos2d-x C++) and concise inline comments using that engine’s API.
2. **Unit tests** — Same code → generate framework-appropriate unit tests (optional second tab).

## Why this helps

Game code in Unity, Pygame, or Cocos2d-x often has no docstrings. This tool **writes docstrings for you** in that engine’s style. You can paste a **single function**, a **method**, or a **whole class** (e.g. a MonoBehaviour with several methods, a Python class, a C++ class). Select your framework so the output uses the right conventions — Pygame (PEP-257), Unity (C# XML summary), or Cocos2d-x (C++ Doxygen-style).

In [13]:
# Imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [14]:
# Setup
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

MODELS = {
    "gpt-4o-mini": "openai/gpt-4o-mini",
    "Llama": "ollama/llama3.2",
}
DEFAULT_MODEL = "gpt-4o-mini"

In [15]:
# Supported frameworks (default: Pygame)
FRAMEWORKS = ["Pygame", "Unity", "Cocos2d-x"]
DEFAULT_FRAMEWORK = "Pygame"

# System prompts per framework: docstrings (and unit tests)
# Input can be a function, method, or class — we add docstrings to whatever the user pastes.
SYSTEM_DOCSTRING = {
    "Pygame": """You are an expert Pygame developer. Your job is to write docstrings for the user's code.

The user's code may be a single function, several functions, or a class with methods (e.g. a sprite class, game loop helpers). It is Pygame/Python (Rect, Surface, Sprite, blit, event loop, collision, etc.).

For each function or method: add a concise PEP-257 docstring (what it does, parameters, return value, side effects). Use Pygame terms: Rect, Surface, Sprite, blit, event loop, collision, velocity, screen. For a class: add a class-level docstring and docstrings for each public method. Add brief inline comments where they help readability; do not state the obvious.

Output only the same code unit with docstrings and comments added. No markdown code fence, no explanation. Do not change variable names or refactor logic.""",
    "Unity": """You are an expert Unity developer. Your job is to write documentation for the user's code.

The user's code may be a single method, several methods, or a whole class (e.g. a MonoBehaviour with Start/Update and other methods). It is Unity/C# (MonoBehaviour, Transform, GameObject, Rigidbody, Collider, Input, etc.).

For each method: add a concise XML doc comment (summary, param, returns). Use Unity terms: Transform, GameObject, Rigidbody2D/Rigidbody, Collider, Vector2/Vector3, Time.deltaTime, Input. For a class: add a class-level summary and document each public method. Add brief inline comments where they help; do not state the obvious.

Output only the same code unit with doc comments and comments added. No markdown code fence, no explanation. Do not change variable names or refactor logic.""",
    "Cocos2d-x": """You are an expert Cocos2d-x developer. Your job is to write docstrings for the user's code.

The user's code may be a single function, several functions, or a class with methods (e.g. a Node subclass, scene logic). It is Cocos2d-x/C++ (Node, Sprite, Director, Scene, Layer, Vec2, Rect, etc.).

For each function or method: add a concise Doxygen-style comment (brief description, param, return). Use Cocos2d-x terms: Node, Sprite, Director, Scene, Layer, Vec2, Size, Rect, schedule, callback. For a class: add a class comment and document each public method. Add brief inline comments where they help; do not state the obvious.

Output only the same code unit with doc comments and comments added. No markdown code fence, no explanation. Do not change variable names or refactor logic.""",
}

SYSTEM_TESTS = {
    "Pygame": """You are an expert Pygame developer and testing expert.
The user's code may be a function, method, or class — Pygame/Python (Rect math, collision, sprite update, movement, drawing helpers).
Your task is to generate pytest unit tests that:
1. Cover typical cases, edge cases (e.g. zero velocity, overlapping Rects, empty groups, off-screen bounds), and invalid inputs where relevant.
2. Use clear test names (e.g. test_rects_collide_when_overlapping, test_movement_clamps_to_screen).
3. For Pygame types: use pygame.Rect, and mock or skip pygame.Surface/pygame.display if needed. Prefer testing pure logic (Rect math, collision detection) without initializing full Pygame. Use pytest. Import pygame if the tests need it.

Output only the test code. No markdown code fence, no explanation. Do not modify the original code.""",
    "Unity": """You are an expert Unity developer and testing expert.
The user's code may be a method or class — Unity/C# (movement, collision, health, input handling).
Your task is to generate unit tests using NUnit or Unity Test Framework (UnityEngine.TestTools) that:
1. Cover typical cases, edge cases (e.g. zero velocity, null references, bounds), and invalid inputs where relevant.
2. Use clear test names (e.g. TestMovementClampsToBounds, TestDamageReducesHealth).
3. Mock or create minimal GameObjects/Transforms/Components as needed. Prefer testing logic that can be tested without full Play mode where possible.

Output only the test code. No markdown code fence, no explanation. Do not modify the original code.""",
    "Cocos2d-x": """You are an expert Cocos2d-x developer and testing expert.
The user's code may be a function or class — Cocos2d-x/C++ (Node logic, collision, movement, Vec2 math).
Your task is to generate unit tests using a C++ test framework (e.g. Google Test, or brief testing instructions if the code is tightly coupled to the engine) that:
1. Cover typical cases, edge cases (e.g. zero velocity, empty nodes, bounds), and invalid inputs where relevant.
2. Use clear test names (e.g. TestVec2Addition, TestCollisionWhenOverlapping).
3. Mock or stub Cocos2d-x types (Node, Director) if needed; prefer testing pure logic (Vec2 math, collision helpers) without full engine init.

Output only the test code (or test code plus short comment on how to run). No markdown code fence, no explanation. Do not modify the original code.""",
}

In [16]:
def strip_code_fence(text: str) -> str:
    if not text:
        return ""
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[-1] if "\n" in t else t[3:]
    if t.endswith("```"):
        t = t.rsplit("```", 1)[0].strip()
    return t

def _system_doc(framework: str):
    return SYSTEM_DOCSTRING.get(framework, SYSTEM_DOCSTRING[DEFAULT_FRAMEWORK])

def _system_tests(framework: str):
    return SYSTEM_TESTS.get(framework, SYSTEM_TESTS[DEFAULT_FRAMEWORK])

def generate_docstring(code: str, model_name: str, framework: str):
    """Stream: add docstring and comments for the selected framework. Yields accumulated output."""
    if not (code and code.strip()):
        yield "Paste your code (function, method, or class) above and select a framework."
        return
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    stream = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": _system_doc(framework)},
            {"role": "user", "content": code},
        ],
        stream=True,
    )
    acc = ""
    for chunk in stream:
        part = chunk.choices[0].delta.content if chunk.choices else None
        if part:
            acc += part
            yield strip_code_fence(acc)

def generate_tests(code: str, model_name: str, framework: str):
    """Stream: generate unit tests for the selected framework. Yields accumulated output."""
    if not (code and code.strip()):
        yield "Paste your code (function, method, or class) above and select a framework."
        return
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    stream = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": _system_tests(framework)},
            {"role": "user", "content": code},
        ],
        stream=True,
    )
    acc = ""
    for chunk in stream:
        part = chunk.choices[0].delta.content if chunk.choices else None
        if part:
            acc += part
            yield strip_code_fence(acc)

In [17]:
# Gradio UI
with gr.Blocks(title="Game Engine Docstring Writer") as app:
    gr.Markdown("Helps **Unity**, **Pygame**, and **Cocos2d-x** programmers write **docstrings** for their code. Select your framework, then paste a **function, method, or class** (e.g. one method, a whole MonoBehaviour, a Python class). Get back the same code with docstrings and comments in that engine’s style.")
    with gr.Row():
        framework_selector = gr.Dropdown(choices=FRAMEWORKS, value=DEFAULT_FRAMEWORK, label="Framework")
        model_selector = gr.Dropdown(choices=list(MODELS.keys()), value=DEFAULT_MODEL, label="Model")

    with gr.Tabs():
        with gr.Tab("Docstrings & comments"):
            gr.Markdown("**Write docstrings** for your code: PEP-257 (Pygame), C# XML docs (Unity), or Doxygen-style (Cocos2d-x). Add a docstring to the class and to each function/method as appropriate.")
            with gr.Row():
                code_in_doc = gr.Code(label="Your code (function, method, or class)", language="python", lines=18)
                code_out_doc = gr.Code(label="Code with docstrings & comments", language="python", lines=18)
            btn_doc = gr.Button("Write docstrings", variant="primary")
            btn_doc.click(generate_docstring, [code_in_doc, model_selector, framework_selector], code_out_doc)

        with gr.Tab("Unit tests"):
            gr.Markdown("Generate unit tests for your code: pytest (Pygame), NUnit/Unity Test (Unity), or C++ tests (Cocos2d-x). Works for functions, methods, or classes.")
            with gr.Row():
                code_in_tests = gr.Code(label="Your code (function, method, or class)", language="python", lines=18)
                code_out_tests = gr.Code(label="Generated unit tests", language="python", lines=18)
            btn_tests = gr.Button("Generate unit tests", variant="primary")
            btn_tests.click(generate_tests, [code_in_tests, model_selector, framework_selector], code_out_tests)

In [ ]:
app.launch()